# CV-Based Naive RAG Application

**Author:** Youssef Sayed  
**Objective:** Build a Retrieval-Augmented Generation system that uses my personal CV as a knowledge base and answers questions based on the information it contains.

---

## Architecture

```
CV Document  -->  Text Splitting  -->  Embedding (OpenAI)  -->  ChromaDB Vector Store
                                                                        |
User Query  ---------------------------------------------------->  Similarity Search
                                                                        |
                                                               Retrieved Chunks (Top-K)
                                                                        |
                                                               Prompt + Context + Query
                                                                        |
                                                                GPT-4o-mini  -->  Answer
```

**Design choices:**
- Splitter: `RecursiveCharacterTextSplitter` (chunk_size=500, overlap=100) to keep CV sections coherent.
- Embeddings: `text-embedding-3-small` — good balance of cost and quality.
- Vector Store: ChromaDB, in-memory.
- LLM: `gpt-4o-mini`, temperature=0 for deterministic answers.
- Retrieval: cosine similarity, top-4 chunks.
- **Scope:** Naive RAG only. No reranking, query expansion, HyDE, or other advanced techniques.

---
## 1. Setup

In [ ]:
# Uncomment to install dependencies
# !pip install langchain langchain-openai langchain-community chromadb tiktoken python-dotenv

In [ ]:
import os
import time
from dotenv import load_dotenv
from IPython.display import display, Markdown

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print('All imports loaded.')

In [ ]:
# Load API key from .env file
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Create a .env file with your key."
print("API key loaded.")

---
## 2. Load the CV

In [ ]:
import os

pdf_path = os.path.join('data', 'Youssef_Sayed-CV.pdf')
txt_path = os.path.join('data', 'youssef_cv.txt')

if os.path.exists(pdf_path):
    loader = PyPDFLoader(pdf_path)
    print(f'Loading from PDF: {pdf_path}')
else:
    loader = TextLoader(txt_path, encoding='utf-8')
    print(f'Loading from text: {txt_path}')

documents = loader.load()
total_chars = sum(len(d.page_content) for d in documents)
print(f'Loaded {len(documents)} document page(s), total characters: {total_chars}')
print('-' * 60)
print(documents[0].page_content[:400].strip())
print('-' * 60)

---
## 3. Chunking

`RecursiveCharacterTextSplitter` tries `["\n\n", "\n", " "]` in order, so it'll naturally split along section boundaries in the CV.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks.")
print()
for i, chunk in enumerate(chunks):
    preview = chunk.page_content[:80].replace("\n", " ")
    print(f"  Chunk {i+1} ({len(chunk.page_content)} chars): {preview}...")

---
## 4. Embedding & Indexing

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="cv_knowledge_base"
)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

print(f"Indexed {len(chunks)} chunks into ChromaDB. Retriever: top-4 similarity search.")

---
## 5. RAG Chain

The prompt is designed to:
- Only use the retrieved context to answer.
- Refuse to answer if the information isn't in the CV.

In [ ]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions about Youssef's CV.
Use ONLY the provided context to answer the question.
If the context does not contain enough information to answer, say:
\"This information is not available in the CV.\"
Do not make up or infer anything not explicitly stated.

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

---
## 6. Query Helper

A wrapper that shows the answer, retrieved chunks, and response time.

In [ ]:
def ask(question, show_context=True):
    """Query the RAG pipeline and display results with diagnostics."""
    
    retrieved_docs = retriever.invoke(question)
    
    start = time.time()
    answer = rag_chain.invoke(question)
    latency = time.time() - start
    
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"   [{latency:.2f}s | {len(retrieved_docs)} chunks retrieved]")
    
    if show_context:
        print()
        for i, doc in enumerate(retrieved_docs):
            preview = doc.page_content[:200].replace("\n", " ")
            print(f"   Chunk {i+1}: {preview}...")
    
    print("-" * 80)
    return {"question": question, "answer": answer, "chunks": len(retrieved_docs), "latency": round(latency, 2)}

---
## 7. Testing & Evaluation

Three categories of questions:

| Category | Goal | Expected behavior |
|---|---|---|
| In-scope | Answer exists in CV | Accurate, factual answer |
| Out-of-scope | Answer is NOT in CV | Refusal / "not available" |
| Edge cases | Broad or ambiguous | Reasonable answer, no hallucination |

### 7.1 In-Scope Questions

In [ ]:
r1 = ask("What university does Youssef attend and what is his current grade?")

In [ ]:
r2 = ask("What programming languages does Youssef know?")

In [ ]:
r3 = ask("Describe the Hakawi project.")

In [ ]:
r4 = ask("What is the Tracera project and what technologies does it use?")

In [ ]:
r5 = ask("What certifications or courses has Youssef completed?")

In [ ]:
r6 = ask("What was Youssef's rank in the ECPC qualifications?")

### 7.2 Out-of-Scope Questions

The system should decline to answer these.

In [ ]:
r7 = ask("What is Youssef's expected salary?")

In [ ]:
r8 = ask("Does Youssef know Rust or Go?")

In [ ]:
r9 = ask("What is the capital of France?")

In [ ]:
r10 = ask("What was Youssef's Thanaweya Amma score?")

In [ ]:
r11 = ask("Does Youssef have any work experience at Google or Microsoft?")

### 7.3 Edge Cases

In [ ]:
r12 = ask("Give a brief summary of Youssef's profile.")

In [ ]:
r13 = ask("Is Youssef a good fit for an ML engineering internship? Why?")

---
## 8. Results Summary

In [ ]:
all_results = [r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12, r13]
categories = ["In-Scope"]*6 + ["Out-of-Scope"]*5 + ["Edge Case"]*2

print(f"{'#':<4} {'Category':<14} {'Latency':<9} {'Question':<55} {'Answer (preview)'}")
print("-" * 130)
for i, (r, cat) in enumerate(zip(all_results, categories), 1):
    q = r['question'][:52]
    a = r['answer'][:60].replace('\n', ' ')
    print(f"{i:<4} {cat:<14} {r['latency']:<9} {q:<55} {a}...")

avg = sum(r['latency'] for r in all_results) / len(all_results)
print(f"\nAverage latency: {avg:.2f}s | Total tests: {len(all_results)}")

---
## 9. Observations

### What works
- Retrieves the correct CV sections for in-scope questions.
- Refuses to answer out-of-scope questions instead of hallucinating.
- Latency is reasonable for a small knowledge base.

### Limitations (expected for Naive RAG)
- No query rewriting, so follow-up / conversational questions won't work well.
- No reranking — chunks are ranked purely by embedding similarity.
- Fixed character-based chunking may split related info across chunks.
- No conversation memory — each question is independent.

### Possible improvements (out of scope for this task)
- Cross-encoder reranking for better precision.
- Query rewriting for conversational context.
- Metadata filtering for section-specific retrieval.
- Contextual compression to reduce token waste.

> These techniques are excluded per the task's scope restriction.